In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

In [2]:
#%load_ext autoreload
#%reload_ext autoreload
#%autoreload 2
#from model_functions import *

%run Model_functions.ipynb

# EMIT FULL BANDS DATA

In [3]:
#EMIT_DATA_CSV         = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT.csv"
EMIT_DATA_CSV         = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EMIT_EO_CANOPY.csv"
#EMIT_MISSING_DATA_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT_MISSING.csv"

In [4]:
emit_df = pd.read_csv(EMIT_DATA_CSV)
print(emit_df.shape)

(3880, 316)


## DATA PREPROCESSING

### Select feature columns

In [5]:
non_feature_cols = [
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'plant_AGB_kg',        # Target variable
    'diameter',            # Allometric
    'height'               # Allometric
]
target = 'plant_AGB_kg'

feature_cols = [c for c in emit_df.columns if c not in non_feature_cols]

X_belige = emit_df[feature_cols]
y_belige = emit_df[target]

X_belige = X_belige.rename({'tandemx_height_m': 'height'}, axis=1)

print(f"Features : {X_belige.columns}")
print(f"Rows     : {len(emit_df)}")

Features : Index(['plot_id', 'species', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'NBR', 'MSI',
       'EVI', 'CIrededge',
       ...
       'EMIT_R1357', 'EMIT_R1417', 'EMIT_R1424', 'EMIT_R1432', 'EMIT_R1774',
       'EMIT_R1781', 'EMIT_R1789', 'EMIT_R1796', 'simard_height_m', 'height'],
      dtype='object', length=304)
Rows     : 3880


## Create interaction terms.

In [6]:
interact_col_names = ['EVI', 'MSI', 'NBR', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']
for col in interact_col_names:
    if col in X_belige.columns:
        X_belige[f'height_x_{col}'] = X_belige['height'] * X_belige[col]

In [7]:
interaction_cols = [c for c in X_belige.columns if c.startswith('height_x_')]
print(X_belige[interaction_cols].corrwith(y_belige).abs().sort_values(ascending=False))

height_x_NBR          0.226619
height_x_CIrededge    0.089816
height_x_NDRE1        0.085668
height_x_NDRE2        0.083697
height_x_MSI          0.081560
height_x_EVI          0.077724
height_x_NDRE3        0.026086
dtype: float64


### Handle EMIT columns with NULL data

In [8]:
X_belige = handle_null_columns(X_belige)

Total NULL count           : 17604
Rows with at least one NULL: 1467
Total rows                 : 3880
Percentage                 : 37.8%

NULL count per column in affected rows:
EMIT_R1432    1467
EMIT_R1350    1467
EMIT_R1417    1467
EMIT_R1424    1467
EMIT_R1342    1467
EMIT_R1774    1467
EMIT_R1781    1467
EMIT_R1789    1467
EMIT_R1796    1467
EMIT_R1335    1467
EMIT_R1327    1467
EMIT_R1357    1467
dtype: int64
Dropping 12 columns:
['EMIT_R1327', 'EMIT_R1335', 'EMIT_R1342', 'EMIT_R1350', 'EMIT_R1357', 'EMIT_R1417', 'EMIT_R1424', 'EMIT_R1432', 'EMIT_R1774', 'EMIT_R1781', 'EMIT_R1789', 'EMIT_R1796']

NULL count after dropping: 0


These 12 bands are all in the 1300–1800 nm range — the shortwave infrared (SWIR) region. EMIT masks out specific wavelength ranges that are dominated by atmospheric water vapor absorption, where the signal is unreliable. The two main atmospheric water vapor absorption windows in EMIT are:  

~1340 to 1460 nm  ← water vapor absorption band  
~1780 to 1970 nm  ← water vapor absorption band  

All of the 12 null columns listed above, fall squarely in these two ranges. EMIT sets these to nodata for pixels where atmospheric correction failed or where the absorption is too strong to recover reliable surface reflectance.  

The above bands are unreliable by design. Drop them.

### Remove Low Variance Features (cols)

### Remove Features With Weak Correlation to Target

**Show correlations**

In [ ]:
numerical_cols = get_numerical_cols(X_belige)

target_corr = plot_correlation_matrix(X_belige[numerical_cols],
                                      y_belige,
                                      top_n=20)
print(target_corr)

**EMIT bands clustered around 0.142**  
 - All EMIT bands show nearly identical correlations in a very tight range of 0.1416 to 0.1427. This is suspicious
 - Genuine spectral variation across 285 bands should produce more spread in correlations
 - The uniformity suggests these bands are highly correlated with each other and likely all capturing the same underlying signal — probably canopy reflectance in the NIR plateau region around 900-1100nm where vegetation reflectance is high and relatively flat
 - The bands shown — EMIT_R865 through EMIT_R1104 — are all in the NIR region. They are essentially measuring the same thing from slightly different wavelengths

**Remove uncorrelated numerical columns**

corr_threshold = 0.05
X_belige = remove_uncorrelated_numerical_cols(X_belige, y_belige,
                                       threshold=corr_threshold,
                                       exclude_cols=['height'])
assert X_belige is not None

X_belige = remove_uncorrelated_categorical_cols(X_belige, y_belige)
assert X_belige is not None

# TEST PARAMETERS

## TEST FEATURES

In [ ]:
struct_features   = ['height']
interaction_terms = [c for c in X_belige.columns if c.startswith('height_x_')]
emit_indices      = ['NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'NBR', 'MSI', 'EVI', 'CIrededge']
emit_cols         = [c for c in X_belige.columns if c.startswith('EMIT')]

# NIR removed — no standalone NIR in EMIT
top_spectral_1 = ['height', 'EVI', 'NBR', 'MSI', 'NDRE1']
top_spectral_2 = ['species', 'height', 'EVI', 'NBR', 'MSI', 'NDRE1']
redband_1      = ['height', 'EVI', 'NBR', 'MSI', 'NDRE1', 'NDRE2', 'CIrededge']
redband_2      = ['species', 'height', 'EVI', 'NBR', 'MSI', 'NDRE1', 'NDRE2', 'CIrededge']

emit_features_list = [
    emit_cols,
    emit_cols + struct_features,
    emit_cols + interaction_terms,

    emit_indices,
    emit_indices + struct_features,
    emit_indices + interaction_terms,

    emit_cols + emit_indices,
    emit_cols + emit_indices + struct_features,
    emit_cols + emit_indices + interaction_terms,

    emit_cols + emit_indices + struct_features,
    emit_cols + emit_indices + struct_features + interaction_terms,

    top_spectral_1,
    top_spectral_1 + struct_features,
    top_spectral_1 + interaction_terms,

    top_spectral_2,
    top_spectral_2 + struct_features,
    top_spectral_2 + interaction_terms,

    redband_1,
    redband_1 + struct_features,
    redband_1 + interaction_terms,

    redband_2,
    redband_2 + struct_features,
    redband_2 + interaction_terms,
]

test_cv = 5

## Create plot groups

In [ ]:
# Retain the groups/plot_id for splitting the data based on groups.
if 'plot_id' in X_belige.columns:
    plot_groups_belige = X_belige['plot_id'].copy()
    X_belige = X_belige.drop(columns=['plot_id'])

site_groups_belige = plot_groups_belige.map(lambda x: x.rsplit('_', maxsplit=1)[0])

near_zero_sites_belige, high_agb_sites_belige,\
    near_zero_plots_belige, high_agb_plots_belige = get_low_and_high_agb_plots(y_belige,
                                                                               plot_groups_belige)

grp_belige = GROUP_INFO(near_zero_sites_belige, high_agb_sites_belige,
                        near_zero_plots_belige, high_agb_plots_belige,
                        groups=plot_groups_belige,
                        cv=test_cv)

In [ ]:
global_experiment_list = {}

# LINEAR REGRESSION

In [ ]:
lin_emit_no_groups = {}
run_experiment(X_belige, y_belige,
               is_groups=False, group_info=grp_belige,
               features_list=emit_features_list,
               model_type="linear",
               is_grid=False,
               is_stratified=False,
               experiments=lin_emit_no_groups,
               global_experiment_list=global_experiment_list)

In [ ]:
lin_emit_groups = {}
run_experiment(X_belige, y_belige,
               is_groups=True, group_info=grp_belige,
               features_list=emit_features_list,
               model_type="linear",
               is_grid=False,
               is_stratified=False,
               experiments=lin_emit_groups,
               global_experiment_list=global_experiment_list)

# RANDOM FOREST.

## No groups

In [ ]:
rf_emit_no_groups_no_grid ={}
run_experiment(X_belige, y_belige,
               is_groups=True, group_info=grp_belige,
               features_list=emit_features_list,
               model_type="rf",
               is_grid=False,
               is_stratified=False,
               experiments=rf_emit_no_groups_no_grid,
               global_experiment_list=global_experiment_list)

In [ ]:
tab_df = tabulate_results(rf_emit_no_groups_no_grid)

## With groups + No tuning

In [ ]:
rf_emit_groups_no_grid = {}
run_experiment(X_belige, y_belige,
               is_groups=True, group_info=grp_belige,
               features_list=emit_features_list,
               model_type="rf",
               is_grid=False,
               is_stratified=False,
               experiments=rf_emit_groups_no_grid,
               global_experiment_list=global_experiment_list)

In [ ]:
tab_df = tabulate_results(rf_emit_groups_no_grid)

## With groups + Tuning

In [ ]:
rf_emit_groups_grid = {}
run_experiment(X_belige, y_belige,
               is_groups=True, group_info=grp_belige,
               features_list=emit_features_list,
               model_type="rf",
               is_grid=False,
               is_stratified=False,
               experiments=rf_emit_groups_grid,
               global_experiment_list=global_experiment_list)

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF EXPERIMENTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list, grouped_only=True)
tab_df = tabulate_results(best_results, grouped_only=True)

## BEST ONE

In [ ]:
best_label, best_result = next(iter(best_results.items()))

print_experiment(best_results, best_label)

In [ ]:
### Residual analysis
y_pred    = best_result["y_pred"]
residuals = best_result["residuals"]

residual_analysis(y_pred, residuals)

# EMIT PCA DATA

In [ ]:
EMIT_PCA_TRAIN_CSV = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/PCA/AGB_EMIT_PCA_TRAIN.csv"
EMIT_PCA_TEST_CSV  = "../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/PCA/AGB_EMIT_PCA_TEST.csv"

In [ ]:
emit_pca_train_df = pd.read_csv(EMIT_PCA_TRAIN_CSV)
emit_pca_test_df  = pd.read_csv(EMIT_PCA_TEST_CSV)

In [ ]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    #'plot_id',             # metadata
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
]

# Drop raw EMIT bands — keep only PCA components
emit_raw_cols = [c for c in emit_pca_train_df.columns if re.match(r'^EMIT_R\d+', c)]

# Training features and target
X_train = emit_pca_train_df.drop(columns=non_feature_cols + emit_raw_cols)
y_train = emit_pca_train_df['plant_AGB_kg']

# Test features and target
X_test  = emit_pca_test_df.drop(columns=non_feature_cols + emit_raw_cols)
y_test  = emit_pca_test_df['plant_AGB_kg']

print(f"Features : {len(X_train.columns)}")
print(f"Columns  : {list(X_train.columns)}")

# Preserve original plot_id for grouping.
groups = pd.concat([X_train, X_test], axis=0)['plot_id']

X_train = X_train.drop(columns=['plot_id'])
X_test = X_test.drop(columns=['plot_id'])

In [ ]:
# One-hot encode species
X_train = pd.get_dummies(X_train, columns=['species'], dtype=int)
X_test  = pd.get_dummies(X_test,  columns=['species'], dtype=int)

# Align columns — test may have different species columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
# Add interaction terms
species_cols = [c for c in X_train.columns if c.startswith('species_')]

for col in species_cols:
    X_train[f'diameter_x_{col}'] = X_train['diameter'] * X_train[col]
    X_test[f'diameter_x_{col}']  = X_test['diameter']  * X_test[col]
    X_train[f'height_x_{col}']   = X_train['height']   * X_train[col]
    X_test[f'height_x_{col}']    = X_test['height']    * X_test[col]

X_train['diameter_x_height'] = X_train['diameter'] * X_train['height']
X_test['diameter_x_height']  = X_test['diameter']  * X_test['height']

In [ ]:
pca_results = {}

## LINEAR REGRESSION ON PCA

## RANDOM FOREST ON PCA

In [ ]:
tabulate_results(pca_results)

**Let us look at the Feature Importances.**  
**Random Forest (Data = Structural variables + Interaction terms):**  
| Feature | Importance |
| :--- | :----: |
  |diameter|                  0.3859 | 
  |diameter_x_Avicennia|      0.2064 |
  |diameter_x_height|         0.1863 | 
  |height|                    0.0803 | 

**Random Forest (Data = Structural variables + EMIT + PCA components):**  
| Feature | Importance |
| :--- | :----: |
  |diameter |                 0.2425  |
  |diameter_x_Avicennia|      0.1877  |
  |diameter_x_height|         0.1822  |
  |height|                    0.0930  |

It appears that adding PCA components is diluting the importance of structural variables — diameter drops from 0.39 to 0.24.  
The model is spreading its attention across more features without gaining predictive power.  
The PCA components are absorbing importance that was previously concentrated in the dominant structural signal.

# FINAL ANALYSIS

**Test R²**  
 — based on one specific split of 12 plots.  
 - A different random seed gives a different number.  
 - Not reproducible or representative.

**CV R² mean**  
 — based on random KFold that splits trees without respecting plot boundaries.  
 - Pseudo-replication inflates it.

It tests on trees, not plots.  
Trees from the same plot appear in both training and test within the same fold.  
The model has already seen the plot — it is not truly unseen.  
The question being answered is "how well does the model predict trees from plots it has partially seen".

**Group CV R² mean**
 — based on 10 folds where entire plots are held out.  
 - Every plot gets to be a test plot exactly once.
 - No pseudo-replication. 
 - Reproducible.
 - This is the number that can answer the question — how well does the model predict AGB for plots it has never seen?

In [ ]:
total_experiments = {**linear_reg_experiments, **random_forest_experiments, **pca_results}
tabulate_results(total_experiments)

**NOTE:**  
Andre's global model achieved R²=0.36 with 524 grid cells and 5 climate variables.  

Without field structural measurements, EO data alone explains essentially none of the individual tree AGB variance across unseen plots in Belize. This is your scientifically honest finding under the sponsor's constraint.  

TanDEM-X height is a stand-level maximum canopy height value. It is the same for every tree in a 12m pixel. It cannot distinguish a 5cm diameter tree from a 40cm diameter tree standing in the same pixel. Field height varies per individual tree and carries real allometric signal. TanDEM-X height does not.  

The allometric equations used in these datasets require diameter and height as inputs.  
Our results show that satellite-derived canopy height and EMIT spectral data cannot predict individual tree AGB without these field measurements.

**plot_id paradox**  
 - Adding plot_id improves EMIT-only performance but hurts or does nothing when structural variables are present.
 - This confirms structural variables already capture what plot_id was proxying for.

**FINALLY**  
 - The above confirms that  diameter, height, and species are the dominant drivers of individual tree AGB.
 - The AGB values in the Belige dataset is computed mathematically using allometric equations. So it is anticipated that the structural parameters like Diameter, height, will be able to explain majority of the AGB variance.

**https://onlinelibrary.wiley.com/doi/10.1002/ldr.70594?af=R**
A study of 109 mangrove trees across eight species in Timor-Leste found very strong positive correlations between DBH and biomass variables (r = 0.963–0.971), indicating that tree diameter represents a reliable predictor of biomass and carbon storage. 

**https://pmc.ncbi.nlm.nih.gov/articles/PMC12425185/**
A study using 302 destructively sampled mangrove trees found that the single parameter with the strongest predictive ability in most studies is diameter at breast height (DBH), and that DBH explains a lot of variability in other tree metrics such as height or aboveground biomass. 

**https://www.sciencedirect.com/science/article/abs/pii/S0304377007001829**
Komiyama et al. (2008) in their widely cited review of mangrove allometry state that trunk diameter of a tree is highly correlated with trunk weight, and that since tree diameter is easy to measure but tree weight is much more difficult to determine, this gives a relatively easy way to estimate biomass. 

**https://www.sciencedirect.com/science/article/abs/pii/S0272771420307022**
Chave et al. (2005) reported that allometric equations with total tree height yielded less biased estimates of AGB, though tree height has often been ignored in carbon-accounting programs because measuring tree height accurately is difficult in mangrove forests. 